<a href="https://colab.research.google.com/github/ucsd-cse-spis-2025/spis25-Project-Andres-Michael/blob/main/Kaggle_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# import libraries
import tensorflow as tf
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

In [2]:
# load the kaggle dataset              from tensorflow

import kagglehub

# Download latest version
path = kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images")

print("Path to dataset files:", path)





Path to dataset files: /kaggle/input/cifake-real-and-ai-generated-synthetic-images


In [3]:
#Sets Train and Test as variables


import os
def get_from_dir(dir):
    dir_path = os.path.join(path, dir)
    print(f"Loading images from: {dir_path}")  # Print the directory path
    if not os.path.isdir(dir_path):
        raise FileNotFoundError(f"Directory not found: {dir_path}")
    return tf.keras.utils.image_dataset_from_directory(
        dir_path,
        labels='inferred',
        color_mode="rgb",
        seed=42,
        batch_size=32,
        image_size=(128, 128) #Can make smaller to make run faster also need to change ln 61 input shape to same size
    )
train=get_from_dir("train")
test=get_from_dir("test")
# print("Structure of x_train dataset:", x_train.element_spec)
# print("Structure of x_test dataset:", x_test.element_spec)

Loading images from: /kaggle/input/cifake-real-and-ai-generated-synthetic-images/train
Found 100000 files belonging to 2 classes.
Loading images from: /kaggle/input/cifake-real-and-ai-generated-synthetic-images/test
Found 20000 files belonging to 2 classes.


In [4]:
# Normalize the images to [0, 1] range
def normalize_img(image, label):
    return tf.cast(image, tf.float32) / 255.0, label

train = train.map(normalize_img)
test = test.map(normalize_img)

In [5]:
# Add a channels dimension
# The add_channels function is not needed as the image_dataset_from_directory function already provides images in (height, width, channels) format.
# def add_channels(image, label):
#     return tf.expand_dims(image, axis=-1), label

# x_train = train.map(add_channels) # Remove this line
# x_test = test.map(add_channels) # Remove this line

# The train and test datasets already have the correct shape after normalization in the previous cell.
x_train = train
x_test = test

print("Structure of x_train dataset:", x_train.element_spec)
print("Structure of x_test dataset:", x_test.element_spec)

Structure of x_train dataset: (TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))
Structure of x_test dataset: (TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))


In [6]:
# Define the CNN model
model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid') # Changed to 1 unit and sigmoid for binary classification
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
# Compile the model
model.compile(optimizer='adam',
              loss=tf.keras.losses.BinaryCrossentropy(),
              metrics=['accuracy'])

In [8]:
# Define the number of samples for validation
validation_split = 0.2  # 20% of the training data (100000 files)
train_size = int((1 - validation_split) * 100000)
validation_size = int(validation_split * 100000)

# Split the training data into training and validation sets before batching
train_ds = train.take(train_size)
val_ds = train.skip(train_size).take(validation_size)

# Apply batching to the split datasets and unpack the elements
# x_train = train_ds.batch(64).map(lambda x, y: (x, y))
# x_val = val_ds.batch(64).map(lambda x, y: (x, y))

# Print the structure of the batched datasets to verify
print("Structure of batched x_train dataset:", train_ds.element_spec)
print("Structure of batched x_val dataset:", val_ds.element_spec)

# You don't need separate y_train and y_val variables when using dataset objects
# The labels are already included within the dataset elements.

Structure of batched x_train dataset: (TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))
Structure of batched x_val dataset: (TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))


In [9]:
# Train the model
history = model.fit(train_ds, epochs=10,
                    validation_data=val_ds)

3125/3125 ━━━━━━━━━━━━━━━━━━━━ 241s 75ms/step - accuracy: 0.7748 - loss: 0.4472


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()
